In [35]:
from langchain_ollama import ChatOllama 

In [36]:
llm=ChatOllama(model="llama3.1:latest")

In [37]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [38]:
# tool creation
@tool
def get_conversion_factor(base_currency:str,target_currency:str)->float:
    """This function fetches the conversion factor"""
    url=f'https://v6.exchangerate-api.com/v6/657e3c6ed6c23a962db6ac1c/pair/{base_currency}/{target_currency}'
    response=requests.get(url)
    return response.json()


@tool
def convert(base_currency_value:int,conversion_rate:float)->float:
    """this function calculates the target currency value from a given base currency value"""
    return base_currency_value*conversion_rate

In [39]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1787961601,
 'time_last_update_utc': 'Sat, 29 Aug 2026 00:00:01 +0000',
 'time_next_update_unix': 1788048001,
 'time_next_update_utc': 'Sun, 30 Aug 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.5338}

In [40]:
convert.invoke({'base_currency_value':10,'conversion_rate':95.5338})

955.338

In [41]:
# tool binding

llm_with_tools=llm.bind_tools([get_conversion_factor,convert])

In [42]:
query=HumanMessage('What is the conversion factor between USD and INR,and based on that can you convert 10 USD to INR')

In [43]:
messages=[query]

In [44]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR,and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={})]

In [45]:
ai_msg=llm_with_tools.invoke(messages)

In [46]:
messages.append(ai_msg)

In [47]:
ai_msg.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'e68bdf91-f18e-4f24-b895-1ef00c6df2e2',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': '10',
   'conversion_rate': 'result_of_get_conversion_factor'},
  'id': 'e52d184a-131f-40f0-8dfe-fc9f9aab363b',
  'type': 'tool_call'}]

In [48]:
import json

In [49]:
for tool_call in ai_msg.tool_calls:
    # find the value of converstion rate using tool 1 and then pass it ot tool 2
    if tool_call['name']=='get_conversion_factor':
        tool_msg1=get_conversion_factor.invoke(tool_call)
        print(tool_msg1)
        # fetch the conversion rate from tool_msg1
        conversion_rate=json.loads(tool_msg1.content)['conversion_rate']

        messages.append(tool_msg1)
    # execute 2nd tool
    if tool_call['name']=='convert':
        tool_call['args']['conversion_rate']=conversion_rate 
        tool_msg2=convert.invoke(tool_call)
        messages.append(tool_msg2)


content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1787961601, "time_last_update_utc": "Sat, 29 Aug 2026 00:00:01 +0000", "time_next_update_unix": 1788048001, "time_next_update_utc": "Sun, 30 Aug 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 95.5338}' name='get_conversion_factor' tool_call_id='e68bdf91-f18e-4f24-b895-1ef00c6df2e2'


In [50]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR,and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:latest', 'created_at': '2026-08-29T19:41:20.817417706Z', 'done': True, 'done_reason': 'stop', 'total_duration': 26232052597, 'load_duration': 7848016418, 'prompt_eval_count': 261, 'prompt_eval_duration': 11151566278, 'eval_count': 55, 'eval_duration': 7231615209, 'logprobs': None, 'model_name': 'llama3.1:latest', 'model_provider': 'ollama'}, id='lc_run--01a04f0a-0fb8-7771-9af2-5aaa0a5b2581-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'e68bdf91-f18e-4f24-b895-1ef00c6df2e2', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': '10', 'conversion_rate': 95.5338}, 'id': 'e52d184a-131f-40f0-8dfe-fc9f9aab363b', 'type': 'tool_call'}], invalid_tool_calls=[],

In [53]:
result=llm_with_tools.invoke(messages) # final result

In [55]:
result.content

'The conversion factor between USD and INR is 1 USD = 95.5338 INR.\n\nConverting 10 USD to INR: 10 * 95.5338 = 955.338 INR.'